In [ ]:
import numpy as np
import pandas as pd
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
import warnings
import time

# Suppress PySR warnings
warnings.filterwarnings("ignore")

# Data generation
def generate_paper_data(degree, n_samples=5000, seed=42):
    """
    Generates exact datasets matching the main NN/DT pipeline.
    Lower degrees match the specific monic/non-monic setups and class mappings.
    """
    rng = np.random.RandomState(seed)
    coef_range = (-10, 10)
    
    if degree == 2:
        # Non-monic: ax^2 + bx + c
        a = rng.uniform(*coef_range, n_samples)
        b = rng.uniform(*coef_range, n_samples)
        c = rng.uniform(*coef_range, n_samples)
        X = np.column_stack([a, b, c])
        discriminant = b**2 - 4*a*c
        y = (discriminant < 0).astype(int) # 0 for real, 1 for complex
        var_names = ['a', 'b', 'c']
        
    elif degree == 3:
        # Monic: x^3 + Ax^2 + Bx + C
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        X = np.column_stack([A, B, C])
        y = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i]])
            y[i] = 1 if np.any(np.abs(roots.imag) > 1e-10) else 0
        var_names = ['a', 'b', 'c']
            
    elif degree == 4:
        # Monic: x^4 + Ax^3 + Bx^2 + Cx + D
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        D = rng.uniform(*coef_range, n_samples)
        X = np.column_stack([A, B, C, D])
        y = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real >= 3: y[i] = 0
            elif n_real >= 1: y[i] = 1
            else: y[i] = 2
        var_names = ['a', 'b', 'c', 'd']
            
    elif degree == 5:
        # Monic: x^5 + Ax^4 + Bx^3 + Cx^2 + Dx + E
        A = rng.uniform(*coef_range, n_samples)
        B = rng.uniform(*coef_range, n_samples)
        C = rng.uniform(*coef_range, n_samples)
        D = rng.uniform(*coef_range, n_samples)
        E = rng.uniform(*coef_range, n_samples)
        X = np.column_stack([A, B, C, D, E])
        y = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real == 5: y[i] = 0
            elif n_real == 3: y[i] = 1
            else: y[i] = 2
        var_names = ['a', 'b', 'c', 'd', 'e']
        
    return X, y, var_names


# Experiment runner
def run_full_robustness_test():
    # Config matching paper constraints
    experiments = [
        {'degree': 2, 'time': 60,  'desc': 'Quadratic'},
        {'degree': 3, 'time': 120, 'desc': 'Cubic'},
        {'degree': 4, 'time': 180, 'desc': 'Quartic'},
        {'degree': 5, 'time': 300, 'desc': 'Quintic'}
    ]
    
    n_trials = 5  # Set to 5 to match Table 2 of the manuscript
    summary_stats = []
    
    print("="*80)
    print(f" SYMBOLIC REGRESSION SUITE (Running {n_trials} trials per Degree)")
    print("="*80)
    
    for exp in experiments:
        deg = exp['degree']
        print(f"\n--- Testing Degree {deg} ({exp['desc']}) ---")
        
        trial_accuracies = []
        best_eq_of_all = ""
        best_acc_of_all = 0.0
        
        # Generate exact dataset used by other models
        X, y, var_names = generate_paper_data(deg, n_samples=5000, seed=42)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        
        # Map class labels (0, 1, 2) back to exact root counts so PySR can use clean integer math
        y_train_roots = deg - (2 * y_train)
        
        for i in range(n_trials):
            start_time = time.time()
            print(f"  Trial {i+1}/{n_trials}...", end=" ", flush=True)
            
            # Initialize PySR
            model = PySRRegressor(
                niterations=200, 
                binary_operators=["+", "-", "*"], 
                unary_operators=["square", "abs", "sign"], 
                timeout_in_seconds=exp['time'],
                maxsize=45,
                population_size=2000,
                model_selection="best", 
                loss="L2DistLoss()",
                parsimony=0.001,
                verbosity=0,
                random_state=i*100 # Distinct seed per trial
            )
            
            # Train on exact root counts
            model.fit(X_train, y_train_roots, variable_names=var_names)
            
            # Predict root counts, then map back to class labels for scoring
            preds_roots = model.predict(X_test)
            preds_classes = (deg - preds_roots) / 2.0
            
            max_class = np.max(y_train)
            y_pred = np.clip(np.round(preds_classes), 0, max_class)
            
            # Calculate Balanced Accuracy
            acc = balanced_accuracy_score(y_test, y_pred)
            trial_accuracies.append(acc)
            
            if acc > best_acc_of_all:
                best_acc_of_all = acc
                best_eq_of_all = model.sympy()
            
            print(f"Acc: {acc:.1%} ({int(time.time() - start_time)}s)")
            
        # Compile Stats
        mean_acc = np.mean(trial_accuracies) * 100
        std_acc = np.std(trial_accuracies) * 100
        
        summary_stats.append({
            'Degree': deg,
            'Accuracy': f"{mean_acc:.1f}% ± {std_acc:.1f}%",
            'Best Eq': str(best_eq_of_all)
        })
        
        print(f"  RESULT: {mean_acc:.1f}% ± {std_acc:.1f}%")
        
    # Print Final Table
    print("\n" + "="*80)
    print(" SYMBOLIC REGRESSION RESULTS SUMMARY")
    print("="*80)
    df = pd.DataFrame(summary_stats)
    print(df.to_string(index=False))
        
    return summary_stats
        
if __name__ == "__main__":
    stats = run_full_robustness_test()